# 04 - NASA C-MAPSS Turbofan Engine Degradation Analysis
Exploratory analysis of flight cycle degradation trajectories in FD001.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
from src.data_loader import load_cmapss

train_df, test_df, rul_df = load_cmapss('FD001')
print(f"Train FD001: {train_df.shape}, Test: {test_df.shape}, RUL ground truth: {rul_df.shape}")
train_df.head()

## Engine Lifespans and Run-to-Failure Distributions

In [ ]:
engine_lifespans = train_df.groupby('unit_nr')['time_cycles'].max()
print(f"Engine lifetime statistics:\nMin: {engine_lifespans.min()} | Max: {engine_lifespans.max()} | Mean: {engine_lifespans.mean():.1f}")

plt.figure(figsize=(9, 4))
sns.histplot(engine_lifespans, bins=20, kde=True, color='#2b5c8f')
plt.title("Distribution of Run-to-Failure Engine Lifespans (FD001)")
plt.xlabel("Total Cycles to Failure")
plt.show()

## Sensor Variance & Degradation Tracking

In [ ]:
sensor_cols = [f's_{i}' for i in range(1, 22)]
stds = train_df[sensor_cols].std()
constant_sensors = stds[stds < 1e-4].index.tolist()
print("Constant sensors with near-zero variance (should be pruned):", constant_sensors)

# Plot key degrading sensors across 5 engines
sample_engines = [1, 2, 3, 4, 5]
plt.figure(figsize=(12, 6))
for eng in sample_engines:
    sub = train_df[train_df['unit_nr'] == eng]
    plt.plot(sub['time_cycles'], sub['s_2'], label=f'Engine #{eng}')
plt.title("Sensor 2 (LPC Outlet Temp) Degradation Curve Across Flight Cycles")
plt.xlabel("Operational Flight Cycles")
plt.ylabel("Sensor 2 Reading")
plt.legend()
plt.show()